# 05 — Inferensi 4 Model ke Seluruh Chat Dota 2 (2016-2026)

Loop atas {BERT, RoBERTa, DistilBERT} × {sentiment, toxicity} + Detoxify (toxicity zero-shot) × seluruh `data/processed/<folder>.parquet`.

Output: `data/inference/<model>_<task>/<folder>.parquet` dengan kolom `match_id, time, player_slot, prob_*, pred_*`.

**Membutuhkan GPU CUDA**. Estimasi 2-8 jam untuk seluruh dekade. Skip-if-exists per-folder supaya bisa di-resume.

**Pra-syarat**: notebook 04 selesai (checkpoint di `models/`).

In [1]:
%pip install detoxify
!pip install detoxify

  Using cached detoxify-0.5.2-py3-none-any.whl.metadata (13 kB)
Using cached detoxify-0.5.2-py3-none-any.whl (12 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Sel 1: Setup
import sys
from pathlib import Path
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'src').is_dir() and (p / 'configs').is_dir()), _here)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import os
os.chdir(_root)

from src.runtime import load_config, print_banner, RunLog
config = load_config('configs/experiment.yaml')
print_banner('05_inference', config)
run_log = RunLog(notebook='05_inference', config_path='configs/experiment.yaml')

import torch
print(f'CUDA: {torch.cuda.is_available()}')

c:\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Notebook: 05_inference
Experiment: thesis-sentiment-toxicity-dota2-decade
Seed: 42  |  Git: be03038
Started at: 2026-05-05T04:56:14+00:00Z
Versi paket:
  - python: 3.10.0
  - transformers: 5.5.0
  - torch: 2.8.0+cu128
  - datasets: 4.8.5
  - scikit-learn: 1.7.1
  - pandas: 2.3.3
  - numpy: 1.26.4
CUDA: True


In [3]:
# Sel 2: Konfigurasi
PROCESSED_ROOT = Path(config['data']['processed_root'])
INFERENCE_ROOT = Path(config['data']['inference_root'])
MODELS_ROOT = Path('models')
BATCH_SIZE = int(config['inference']['batch_size'])
FP16 = config['inference']['precision'] == 'fp16'

FT_MODELS = ['bert', 'roberta', 'distilbert']

processed_files = sorted([p for p in PROCESSED_ROOT.glob('*.parquet') if not p.stem.endswith('_non_english')])
print(f'Processed folders: {len(processed_files)}')
for p in processed_files:
    print(f'  - {p.name}')

Processed folders: 14
  - 2016.parquet
  - 2017.parquet
  - 2018.parquet
  - 2019.parquet
  - 2020.parquet
  - 2021.parquet
  - 2022.parquet
  - 2023.parquet
  - 2024.parquet
  - 2025.parquet
  - 202601.parquet
  - 202602.parquet
  - 202603.parquet
  - 202604.parquet


In [4]:
# Sel 3: Inferensi 3 model fine-tuned × 2 tugas
from src.inference.batch_inference import infer_folder

for model_key in FT_MODELS:
    for task in ['sentiment', 'toxicity']:
        model_dir = MODELS_ROOT / f'{model_key}-{task}'
        if not model_dir.exists():
            msg = f'Skip {model_key}/{task} — checkpoint tidak ada di {model_dir}'
            print(f'[WARN] {msg}')
            run_log.add_warning(msg)
            continue
        out_root = INFERENCE_ROOT / f'{model_key}_{task}'
        out_root.mkdir(parents=True, exist_ok=True)
        print(f'\n=== {model_key} / {task} ({model_dir}) ===')
        for p in processed_files:
            out_path = out_root / p.name
            stats = infer_folder(
                processed_path=p,
                model_dir=model_dir,
                task=task,
                out_path=out_path,
                batch_size=BATCH_SIZE,
                fp16=FP16,
            )
            tag = '[SKIP]' if stats.skipped else '[OK]'
            print(f'  {tag} {p.name}: n={stats.n_output:,} → {stats.out_path}')
            run_log.add_output(stats.out_path)


=== bert / sentiment (models\bert-sentiment) ===


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2618.00it/s]


  [OK] 2016.parquet: n=120,454 → data\inference\bert_sentiment\2016.parquet


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7142.66it/s]


  [OK] 2017.parquet: n=101,812 → data\inference\bert_sentiment\2017.parquet


...

  [OK] 202602.parquet: n=13,442 → data\inference\distilbert_toxicity\202602.parquet


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 7957.67it/s]


  [OK] 202603.parquet: n=14,087 → data\inference\distilbert_toxicity\202603.parquet


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 9783.29it/s]


  [OK] 202604.parquet: n=9,353 → data\inference\distilbert_toxicity\202604.parquet


In [5]:
# Sel 4: Inferensi Detoxify (zero-shot, tanpa fine-tune)
from src.inference.batch_inference import infer_detoxify

out_root = INFERENCE_ROOT / 'detoxify_toxicity'
out_root.mkdir(parents=True, exist_ok=True)
print(f'\n=== detoxify / toxicity (pretrained, zero-shot) ===')
for p in processed_files:
    out_path = out_root / p.name
    stats = infer_detoxify(
        processed_path=p,
        out_path=out_path,
        batch_size=BATCH_SIZE,
    )
    tag = '[SKIP]' if stats.skipped else '[OK]'
    print(f'  {tag} {p.name}: n={stats.n_output:,}')
    run_log.add_output(stats.out_path)


=== detoxify / toxicity (pretrained, zero-shot) ===
Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.1-alpha/toxic_original-c1212f89.ckpt" to C:\Users\user/.cache\torch\hub\checkpoints\toxic_original-c1212f89.ckpt


100%|██████████| 418M/418M [00:22<00:00, 19.2MB/s] 
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1264.49it/s]
BertForSequenceClassification LOAD REPORT from: None
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [OK] 2016.parquet: n=120,454


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2754.32it/s]
BertForSequenceClassification LOAD REPORT from: None
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [OK] 2017.parquet: n=101,812


...

  [OK] 202602.parquet: n=13,442


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3788.76it/s]
BertForSequenceClassification LOAD REPORT from: None
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [OK] 202603.parquet: n=14,087


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2854.78it/s]
BertForSequenceClassification LOAD REPORT from: None
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [OK] 202604.parquet: n=9,353


In [6]:
# Sel 5: Verifikasi join lossless inference × processed
import pandas as pd

# Cek satu kombinasi sebagai sanity check.
for model_key in FT_MODELS:
    for task in ['sentiment', 'toxicity']:
        out_root = INFERENCE_ROOT / f'{model_key}_{task}'
        if not out_root.exists():
            continue
        inf_files = sorted(out_root.glob('*.parquet'))
        proc_total = sum(len(pd.read_parquet(p)) for p in processed_files)
        inf_total = sum(len(pd.read_parquet(p)) for p in inf_files)
        if proc_total != inf_total:
            msg = f'Mismatch {model_key}/{task}: processed={proc_total:,} vs inference={inf_total:,}'
            print(f'[WARN] {msg}')
            run_log.add_warning(msg)
        else:
            print(f'  OK {model_key}/{task}: {inf_total:,} baris cocok dengan processed')

  OK bert/sentiment: 1,603,567 baris cocok dengan processed
  OK bert/toxicity: 1,603,567 baris cocok dengan processed
  OK roberta/sentiment: 1,603,567 baris cocok dengan processed
  OK roberta/toxicity: 1,603,567 baris cocok dengan processed
  OK distilbert/sentiment: 1,603,567 baris cocok dengan processed
  OK distilbert/toxicity: 1,603,567 baris cocok dengan processed


In [7]:
run_log.save('reports/run_log.csv')

[run_log] 05_inference → 4145.84s, 98 outputs, 0 warnings → reports\run_log.csv
